In [ ]:
from nsbtools.plotting import plot_brain_plotly, update_trace_type, fetch_trace
from nsbtools.io import load_data, read_surf
from matplotlib import pyplot as plt
import plotly.io as pio
import nibabel as nib

pio.renderers.default = "sphinx_gallery"

In [ ]:
# from neuromaps.datasets import fetch_fslr

species = 'human'
den = "32k"

surf_lh_data = load_data("surf", density=den, hemi="L", species=species)
surf_rh_data = load_data("surf", density=den, hemi="R", species=species)
surf_lh = {"v": surf_lh_data.vertices, "t": surf_lh_data.faces}
surf_rh = {"v": surf_rh_data.vertices, "t": surf_rh_data.faces}

medmask_lh = load_data("medmask", density=den, hemi="L", species=species)
medmask_rh = load_data("medmask", density=den, hemi="R", species=species)

data_lh = load_data("fcgradient1", density=den, hemi="L")
# sulc_data = fetch_fslr(den)["sulc"]
# data_lh = nib.load(sulc_data[0]).darrays[0].data
# data_rh = nib.load(sulc_data[1]).darrays[0].data * 10 

parc_lh_file = f"/Users/victorbarnes/phd_local/HeteroModes/data/parcellations/parc-hcpmmp1_space-fsLR_den-{den}_hemi-L.label.gii"
parc_lh = nib.load(parc_lh_file).darrays[0].data.astype(int)

In [ ]:
# Your time-varying data
nverts = len(medmask_lh)
ntimepoints = 10
activity = np.random.randn(nverts, ntimepoints)  # shape: (nverts, ntimepoints)

In [ ]:
surfs = {"lh": surf_lh}
data = {"lh": activity[:, 0]}
rois = {"lh": medmask_lh}

fig_animate = plot_brain_plotly(surfs, data=data, rois=rois, layout="row", views=["lateral"], 
                         mesh_edges=False, size=(600, 400), zoom=2, cbar=False, roi_outlines=True, cmap="turbo")
# Add colorbar manually
# fig2.update_layout(
#     showscale=True,
# )

fig_animate.show()

In [ ]:
from plotly import graph_objects as go

frames = []
for t in range(ntimepoints):
    frame_data = []
    
    # You need to update each Mesh3d trace in your figure
    # The order matches the traces added in plot_brain_plotly
    for trace_idx, trace in enumerate(fig.data):
        if isinstance(trace, go.Mesh3d) and hasattr(trace, 'intensity'):
            # Update intensity with new timepoint data
            frame_data.append(
                go.Mesh3d(intensity=activity[:, t])
            )
        else:
            # Keep other traces (edges, mask) unchanged
            frame_data.append(trace)
    
    frames.append(
        go.Frame(
            data=frame_data,
            name=f't{t}',
            traces=list(range(len(fig_animate.data)))  # Which traces to update
        )
    )

fig_animate.frames = frames

In [ ]:
fig_animate.update_layout(
    updatemenus=[{
        'type': 'buttons',
        'showactive': False,
        'buttons': [
            {
                'label': 'Play',
                'method': 'animate',
                'args': [None, {
                    'frame': {'duration': 100, 'redraw': True},
                    'fromcurrent': True,
                    'mode': 'immediate',
                    'transition': {'duration': 0}
                }]
            },
            {
                'label': 'Pause',
                'method': 'animate',
                'args': [[None], {
                    'frame': {'duration': 0, 'redraw': False},
                    'mode': 'immediate',
                    'transition': {'duration': 0}
                }]
            }
        ],
        'x': 0.1, 'y': 0, 'xanchor': 'right', 'yanchor': 'top'
    }],
    sliders=[{
        'active': 0,
        'steps': [
            {
                'args': [[f't{t}'], {
                    'frame': {'duration': 0, 'redraw': True},
                    'mode': 'immediate',
                    'transition': {'duration': 0}
                }],
                'label': f'{t}',
                'method': 'animate'
            }
            for t in range(ntimepoints)
        ],
        'x': 0.1, 'len': 0.9, 'xanchor': 'left', 'y': 0,
        'yanchor': 'top', 'pad': {'b': 10, 't': 50}
    }]
)

fig_animate.show()